# Phase 3.6 — SHAP Feature Importance Analysis

SHAP (SHapley Additive exPlanations) shows which operating conditions most influence each temperature prediction. This makes our Normal Behavior Models transparent and explainable.

**What we learn:**
- Which inputs drive each subsystem's predicted temperature
- Whether the relationship is positive (feature pushes prediction up) or negative
- How feature importance varies across subsystems and farms

**Method:** TreeExplainer on LightGBM models — exact SHAP values, computed on a 1000-row sample per farm.

In [1]:
# --- Imports & setup ---
import sys, os, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Install shap if needed
try:
    import shap
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap", "-q"])
    import shap

# Project paths
PROJECT_ROOT = Path(os.path.abspath("")).parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.nbm_config import get_nbm_config, NBM_TARGETS
from src.data.load_data import load_farm_training_data
from src.features.operating_conditions import FEATURE_DESCRIPTIONS

DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = DATA_DIR / "processed" / "models"
FIG_DIR = PROJECT_ROOT / "outputs" / "figures"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"
FIG_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

FARMS = ["Wind Farm A", "Wind Farm B", "Wind Farm C"]
FARM_KEYS = {"Wind Farm A": "a", "Wind Farm B": "b", "Wind Farm C": "c"}
SAMPLE_N = 1000
RANDOM_STATE = 42

print(f"shap version: {shap.__version__}")
print(f"Models dir: {MODEL_DIR}")
print(f"Figures dir: {FIG_DIR}")

C:\Users\talha\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


shap version: 0.51.0
Models dir: D:\Personal Projects\Enbridge Case Compettion\data\processed\models
Figures dir: D:\Personal Projects\Enbridge Case Compettion\outputs\figures


In [2]:
# --- Helper functions ---

def get_human_names(farm: str, feature_cols: list[str]) -> list[str]:
    """Convert raw column names to human-readable labels."""
    desc = FEATURE_DESCRIPTIONS[farm]
    names = []
    for col in feature_cols:
        if col in desc:
            label, unit = desc[col]
            names.append(f"{label} ({unit})")
        else:
            names.append(col)
    return names


def load_model(farm_key: str, subsystem: str):
    """Load a trained LightGBM NBM model."""
    path = MODEL_DIR / f"farm_{farm_key}" / f"{subsystem}_nbm.joblib"
    return joblib.load(path)


def compute_shap_values(model, X_sample, feature_names):
    """Compute SHAP values using TreeExplainer."""
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    return shap_values, explainer


def plot_beeswarm(shap_values, X_sample, feature_names, title, save_path):
    """Create and save a SHAP beeswarm (summary) plot."""
    fig, ax = plt.subplots(figsize=(12, 7))
    shap.summary_plot(
        shap_values, X_sample,
        feature_names=feature_names,
        show=False,
        plot_size=None,
    )
    plt.title(title, fontsize=14, fontweight="bold", pad=15)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")
    print(f"  Saved: {save_path.name}")


# Storage for all SHAP importance data (for cross-farm analysis later)
all_shap_importance = {}  # farm -> subsystem -> {feature: mean_abs_shap}

print("Helper functions ready.")

Helper functions ready.


## Farm A — Feature Importance Per Subsystem

Farm A has **5 NBM models** (gearbox, generator bearings, transformer, hydraulic, cooling) with **8 input features**. The beeswarm plots below show which operating conditions drive each temperature prediction.

In [3]:
# --- Farm A: Load data & compute SHAP for each subsystem ---
farm = "Wind Farm A"
farm_key = FARM_KEYS[farm]
config = get_nbm_config(farm)
input_cols = config["inputs"]
human_names = get_human_names(farm, input_cols)

print(f"Loading training data for {farm}...")
df_train = load_farm_training_data("A")
X_all = df_train[input_cols].dropna()
X_sample = X_all.sample(n=min(SAMPLE_N, len(X_all)), random_state=RANDOM_STATE)
print(f"  Full training set: {len(X_all):,} rows | Sample: {len(X_sample):,} rows")
print(f"  Input features ({len(input_cols)}): {human_names}\n")

all_shap_importance[farm] = {}
farm_a_shap = {}  # subsystem -> shap_values (for combined plot)

for subsystem, target_col in config["targets"].items():
    target_info = NBM_TARGETS[farm][subsystem]
    print(f"--- {subsystem.upper()} (target: {target_info['description']}) ---")
    
    model = load_model(farm_key, subsystem)
    shap_vals, explainer = compute_shap_values(model, X_sample, human_names)
    farm_a_shap[subsystem] = shap_vals
    
    # Store mean |SHAP| for each feature
    mean_abs = np.abs(shap_vals).mean(axis=0)
    importance_dict = {name: float(val) for name, val in zip(human_names, mean_abs)}
    all_shap_importance[farm][subsystem] = importance_dict
    
    # Beeswarm plot
    save_path = FIG_DIR / f"shap_beeswarm_farm_a_{subsystem}.png"
    plot_beeswarm(
        shap_vals, X_sample, human_names,
        f"Farm A — {subsystem.replace('_', ' ').title()} NBM\n(Target: {target_info['description']})",
        save_path
    )
    print()

Loading training data for Wind Farm A...


  Full training set: 464,694 rows | Sample: 1,000 rows
  Input features (8): ['Windspeed (m/s)', 'Estimated windspeed (m/s)', 'Possible grid active power (kW)', 'Grid power (kW)', 'Ambient temperature (°C)', 'Generator rpm (rpm)', 'Rotor rpm (rpm)', 'Pitch angle (°)']

--- GEARBOX (target: Temperature oil in gearbox) ---


  Saved: shap_beeswarm_farm_a_gearbox.png

--- GENERATOR_BEARINGS (target: Temperature in generator bearing 2 (Drive End)) ---


  Saved: shap_beeswarm_farm_a_generator_bearings.png

--- TRANSFORMER (target: Temperature in HV transformer phase L1) ---


  Saved: shap_beeswarm_farm_a_transformer.png

--- HYDRAULIC (target: Temperature oil in hydraulic group) ---


  Saved: shap_beeswarm_farm_a_hydraulic.png

--- COOLING (target: Temperature in the VCS cooling water) ---


  Saved: shap_beeswarm_farm_a_cooling.png



In [4]:
# --- Farm A: Combined bar chart — mean |SHAP| across all 5 models ---
subsystems = list(farm_a_shap.keys())
n_features = len(human_names)

# Build DataFrame: rows=features, columns=subsystems
importance_df = pd.DataFrame(index=human_names)
for sub in subsystems:
    mean_abs = np.abs(farm_a_shap[sub]).mean(axis=0)
    importance_df[sub.replace("_", " ").title()] = mean_abs

# Sort by total importance across all subsystems
importance_df["Total"] = importance_df.sum(axis=1)
importance_df = importance_df.sort_values("Total", ascending=True)
importance_df = importance_df.drop(columns="Total")

fig, ax = plt.subplots(figsize=(14, 7))
importance_df.plot(kind="barh", ax=ax, width=0.75)
ax.set_xlabel("Mean |SHAP Value| (avg impact on predicted temperature)", fontsize=12)
ax.set_title("Farm A — Feature Importance Across All Subsystems", fontsize=14, fontweight="bold")
ax.legend(title="Subsystem", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
save_path = FIG_DIR / "shap_combined_importance_farm_a.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")
print(f"Saved: {save_path.name}")

Saved: shap_combined_importance_farm_a.png


## Farm B — Feature Importance

Farm B has **3 NBM models** (gearbox, generator bearings, transformer) with **10 input features**.

In [5]:
# --- Farm B: Load data & compute SHAP for each subsystem ---
farm = "Wind Farm B"
farm_key = FARM_KEYS[farm]
config = get_nbm_config(farm)
input_cols = config["inputs"]
human_names_b = get_human_names(farm, input_cols)

print(f"Loading training data for {farm}...")
df_train_b = load_farm_training_data("B")
X_all_b = df_train_b[input_cols].dropna()
X_sample_b = X_all_b.sample(n=min(SAMPLE_N, len(X_all_b)), random_state=RANDOM_STATE)
print(f"  Full training set: {len(X_all_b):,} rows | Sample: {len(X_sample_b):,} rows")
print(f"  Input features ({len(input_cols)}): {human_names_b}\n")

all_shap_importance[farm] = {}

for subsystem, target_col in config["targets"].items():
    target_info = NBM_TARGETS[farm][subsystem]
    print(f"--- {subsystem.upper()} (target: {target_info['description']}) ---")
    
    model = load_model(farm_key, subsystem)
    shap_vals, explainer = compute_shap_values(model, X_sample_b, human_names_b)
    
    # Store mean |SHAP|
    mean_abs = np.abs(shap_vals).mean(axis=0)
    importance_dict = {name: float(val) for name, val in zip(human_names_b, mean_abs)}
    all_shap_importance[farm][subsystem] = importance_dict
    
    # Beeswarm plot
    save_path = FIG_DIR / f"shap_beeswarm_farm_b_{subsystem}.png"
    plot_beeswarm(
        shap_vals, X_sample_b, human_names_b,
        f"Farm B — {subsystem.replace('_', ' ').title()} NBM\n(Target: {target_info['description']})",
        save_path
    )
    print()

Loading training data for Wind Farm B...


  Full training set: 418,792 rows | Sample: 1,000 rows
  Input features (10): ['Wind speed 1 (m/s)', 'Wind speed 2 (m/s)', 'Wind speed (m/s)', 'Active power (kW)', 'Available power (kW)', 'Outside temperature (°C)', 'Generator converter rotational speed (rpm)', 'Gearbox rotational speed (rpm)', 'Rotor speed (rpm)', 'Pitch angle (°)']

--- GEARBOX (target: Gearbox oil tank temperature) ---


  Saved: shap_beeswarm_farm_b_gearbox.png

--- GENERATOR_BEARINGS (target: Generator bearing temperature 1) ---


  Saved: shap_beeswarm_farm_b_generator_bearings.png

--- TRANSFORMER (target: Transformator core temperature) ---


  Saved: shap_beeswarm_farm_b_transformer.png



## Farm C — Feature Importance

Farm C has **5 NBM models** (gearbox, generator bearings, transformer, hydraulic, cooling) with **13 input features**.

In [6]:
# --- Farm C: Load data & compute SHAP for each subsystem ---
farm = "Wind Farm C"
farm_key = FARM_KEYS[farm]
config = get_nbm_config(farm)
input_cols = config["inputs"]
human_names_c = get_human_names(farm, input_cols)

print(f"Loading training data for {farm}...")
df_train_c = load_farm_training_data("C")
X_all_c = df_train_c[input_cols].dropna()
X_sample_c = X_all_c.sample(n=min(SAMPLE_N, len(X_all_c)), random_state=RANDOM_STATE)
print(f"  Full training set: {len(X_all_c):,} rows | Sample: {len(X_sample_c):,} rows")
print(f"  Input features ({len(input_cols)}): {human_names_c}\n")

all_shap_importance[farm] = {}

for subsystem, target_col in config["targets"].items():
    target_info = NBM_TARGETS[farm][subsystem]
    print(f"--- {subsystem.upper()} (target: {target_info['description']}) ---")
    
    model = load_model(farm_key, subsystem)
    shap_vals, explainer = compute_shap_values(model, X_sample_c, human_names_c)
    
    # Store mean |SHAP|
    mean_abs = np.abs(shap_vals).mean(axis=0)
    importance_dict = {name: float(val) for name, val in zip(human_names_c, mean_abs)}
    all_shap_importance[farm][subsystem] = importance_dict
    
    # Beeswarm plot
    save_path = FIG_DIR / f"shap_beeswarm_farm_c_{subsystem}.png"
    plot_beeswarm(
        shap_vals, X_sample_c, human_names_c,
        f"Farm C — {subsystem.replace('_', ' ').title()} NBM\n(Target: {target_info['description']})",
        save_path
    )
    print()

Loading training data for Wind Farm C...


  Full training set: 1,438,937 rows | Sample: 1,000 rows
  Input features (13): ['Wind speed 1 (m/s)', 'Wind speed 1+2 (m/s)', 'Wind speed 2 (m/s)', 'Active power HV grid (kW)', 'Active power grid side converter (kW)', 'Ambient temperature (°C)', 'Rotor speed 1 (1/min)', 'Rotor speed 2 (1/min)', 'Generator angle speed (rad/s)', 'Min pitch angle (°)', 'Position rotor blade axis 1 (°)', 'Position rotor blade axis 2 (°)', 'Position rotor blade axis 3 (°)']

--- GEARBOX (target: Gearbox oil temperature 1) ---


  Saved: shap_beeswarm_farm_c_gearbox.png

--- GENERATOR_BEARINGS (target: Rotor bearing temperature 1) ---


  Saved: shap_beeswarm_farm_c_generator_bearings.png

--- TRANSFORMER (target: Oil temperature 1 main transformer) ---


  Saved: shap_beeswarm_farm_c_transformer.png

--- HYDRAULIC (target: Hydraulic oil tank temperature 1) ---


  Saved: shap_beeswarm_farm_c_hydraulic.png

--- COOLING (target: Cooling water temp. generator inlet 1) ---


  Saved: shap_beeswarm_farm_c_cooling.png



## Cross-Farm Feature Importance Comparison

This heatmap shows which operating conditions matter most for which subsystems across all three farms. Rows are normalized feature categories (since raw column names differ per farm), and columns are farm-subsystem pairs.

In [7]:
# --- Cross-Farm Heatmap: feature importance across all models ---
# Map human-readable names to normalized categories for cross-farm comparison
CATEGORY_MAP = {
    # Wind speed variants
    "Windspeed (m/s)": "Wind Speed",
    "Estimated windspeed (m/s)": "Wind Speed",
    "Wind speed 1 (m/s)": "Wind Speed",
    "Wind speed 2 (m/s)": "Wind Speed",
    "Wind speed (m/s)": "Wind Speed",
    "Wind speed 1+2 (m/s)": "Wind Speed",
    # Power variants
    "Possible grid active power (kW)": "Active Power",
    "Grid power (kW)": "Active Power",
    "Active power (kW)": "Active Power",
    "Available power (kW)": "Active Power",
    "Active power HV grid (kW)": "Active Power",
    "Active power grid side converter (kW)": "Active Power",
    # Ambient temperature
    "Ambient temperature (\u00b0C)": "Ambient Temp",
    "Outside temperature (\u00b0C)": "Ambient Temp",
    # Rotational speeds
    "Generator rpm (rpm)": "Generator Speed",
    "Rotor rpm (rpm)": "Rotor Speed",
    "Generator converter rotational speed (rpm)": "Generator Speed",
    "Gearbox rotational speed (rpm)": "Gearbox Speed",
    "Rotor speed (rpm)": "Rotor Speed",
    "Rotor speed 1 (1/min)": "Rotor Speed",
    "Rotor speed 2 (1/min)": "Rotor Speed",
    "Generator angle speed (rad/s)": "Generator Speed",
    # Pitch angle
    "Pitch angle (\u00b0)": "Pitch Angle",
    "Min pitch angle (\u00b0)": "Pitch Angle",
    "Position rotor blade axis 1 (\u00b0)": "Pitch Angle",
    "Position rotor blade axis 2 (\u00b0)": "Pitch Angle",
    "Position rotor blade axis 3 (\u00b0)": "Pitch Angle",
}

# Aggregate by category: sum mean|SHAP| for features in same category
heatmap_data = {}  # (farm_short, subsystem) -> {category: total_shap}

for farm, subsystems_dict in all_shap_importance.items():
    farm_short = farm.replace("Wind Farm ", "")
    for subsystem, feat_importances in subsystems_dict.items():
        col_label = f"{farm_short}-{subsystem.replace('_', ' ').title()}"
        cat_totals = {}
        for feat_name, shap_val in feat_importances.items():
            cat = CATEGORY_MAP.get(feat_name, feat_name)
            cat_totals[cat] = cat_totals.get(cat, 0) + shap_val
        heatmap_data[col_label] = cat_totals

# Build heatmap DataFrame
heatmap_df = pd.DataFrame(heatmap_data).fillna(0)

# Sort rows by total importance
heatmap_df["Total"] = heatmap_df.sum(axis=1)
heatmap_df = heatmap_df.sort_values("Total", ascending=False)
heatmap_df = heatmap_df.drop(columns="Total")

# Plot heatmap
fig, ax = plt.subplots(figsize=(16, 8))
im = ax.imshow(heatmap_df.values, cmap="YlOrRd", aspect="auto")

# Labels
ax.set_xticks(range(len(heatmap_df.columns)))
ax.set_xticklabels(heatmap_df.columns, rotation=45, ha="right", fontsize=10)
ax.set_yticks(range(len(heatmap_df.index)))
ax.set_yticklabels(heatmap_df.index, fontsize=11)

# Annotate cells with values
for i in range(len(heatmap_df.index)):
    for j in range(len(heatmap_df.columns)):
        val = heatmap_df.values[i, j]
        if val > 0:
            color = "white" if val > heatmap_df.values.max() * 0.6 else "black"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8, color=color)

plt.colorbar(im, ax=ax, label="Mean |SHAP Value| (sum per category)", shrink=0.8)
ax.set_title("Cross-Farm Feature Importance Heatmap\n(Operating Condition Categories vs. Subsystem NBMs)",
             fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
save_path = FIG_DIR / "shap_cross_farm_heatmap.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")
print(f"Saved: {save_path.name}")

Saved: shap_cross_farm_heatmap.png


In [8]:
# --- Save SHAP importance summary report ---
report = {
    "description": "SHAP feature importance summary for all NBM models",
    "sample_size": SAMPLE_N,
    "method": "TreeExplainer (exact SHAP values for LightGBM)",
    "farms": {}
}

for farm, subsystems_dict in all_shap_importance.items():
    farm_report = {}
    for subsystem, feat_importances in subsystems_dict.items():
        # Sort features by importance (descending)
        sorted_feats = sorted(feat_importances.items(), key=lambda x: x[1], reverse=True)
        farm_report[subsystem] = {
            "target": NBM_TARGETS[farm][subsystem]["description"],
            "feature_ranking": [
                {"rank": i+1, "feature": name, "mean_abs_shap": round(val, 4)}
                for i, (name, val) in enumerate(sorted_feats)
            ],
            "top_3": [name for name, _ in sorted_feats[:3]]
        }
    report["farms"][farm] = farm_report

save_path = REPORT_DIR / "shap_importance_summary.json"
with open(save_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"Saved SHAP summary report: {save_path}")
print(f"\nTotal models analyzed: {sum(len(v) for v in all_shap_importance.values())}")

# Print top-3 features per model
for farm, subsystems_dict in report["farms"].items():
    print(f"\n{farm}:")
    for subsystem, info in subsystems_dict.items():
        top3 = ", ".join(info["top_3"])
        print(f"  {subsystem:25s} -> Top 3: {top3}")

Saved SHAP summary report: D:\Personal Projects\Enbridge Case Compettion\outputs\reports\shap_importance_summary.json

Total models analyzed: 13

Wind Farm A:
  gearbox                   -> Top 3: Rotor rpm (rpm), Windspeed (m/s), Grid power (kW)
  generator_bearings        -> Top 3: Rotor rpm (rpm), Ambient temperature (°C), Windspeed (m/s)
  transformer               -> Top 3: Windspeed (m/s), Generator rpm (rpm), Ambient temperature (°C)
  hydraulic                 -> Top 3: Generator rpm (rpm), Ambient temperature (°C), Windspeed (m/s)
  cooling                   -> Top 3: Ambient temperature (°C), Rotor rpm (rpm), Generator rpm (rpm)

Wind Farm B:
  gearbox                   -> Top 3: Rotor speed (rpm), Pitch angle (°), Gearbox rotational speed (rpm)
  generator_bearings        -> Top 3: Active power (kW), Generator converter rotational speed (rpm), Rotor speed (rpm)
  transformer               -> Top 3: Outside temperature (°C), Generator converter rotational speed (rpm), Active 

## Key Observations

**Patterns to look for in the beeswarm plots above:**

1. **Ambient temperature** is expected to rank highly for most subsystems -- it sets the environmental baseline that all internal temperatures respond to.

2. **Active power / wind speed** should be strong drivers for gearbox and generator bearing models, since mechanical load directly generates heat.

3. **Pitch angle** may matter more for hydraulic models (pitch actuators are hydraulically driven) and less for transformer models.

4. **Generator/rotor speed** features capture rotational dynamics -- important for bearing and gearbox thermal behavior.

5. **Cross-farm consistency:** If the same feature categories dominate across farms, it validates that our NBM architecture captures physically meaningful relationships rather than farm-specific artifacts.

**Red (high feature value) pushing prediction up** = physically intuitive (e.g., higher ambient temp -> higher predicted gearbox oil temp).

**Blue (low feature value) pushing prediction up** = potentially interesting non-linear relationship worth investigating.